# Testlauf 2: A100 – Rot auf Blau, mit automatischen Backups

**192 geplante Episoden: 32 Startlagen × 6 Modelle.** Dieselben Positionen und Seeds für alle Modelle; ausschließlich Rot auf Blau. Dory bleibt wegen des noch nicht validierten Fine-Tuning-Pfads ausgeschlossen.

Wähle eine **A100**. Hinterlege deinen HF-Token links unter **Secrets (Schlüsselsymbol)** als `HF_TOKEN` und erlaube dem Notebook den Zugriff. Alternativ erscheint ein verdecktes Eingabefeld. Der Token braucht Schreibzugriff auf das unten angegebene Dataset. Der Token gehört in dieses Feld/Secret, nicht in eine Codezelle.

**Vor dem Modellstart:** automatischer HF-Login und ein echter Schreib-/Lese-/Löschtest. Danach werden der Plan, Startmarkierungen und jeder fertige Versuch zusätzlich zu Drive automatisch auf HF gesichert. Du musst die sechs Stunden nicht erreichen: Nach einem Laufzeitverlust dieselbe Notebook-Version mit denselben Backup-Pfaden starten; bereits begonnene Versuche werden erhalten und nicht wiederholt.

Die anschließende Auswertung und Modellauswahl nimmst du selbst vor. Es entstehen keine vorab behaupteten Modellergebnisse.

In [ ]:
REPO_REF = "dd383c2b1494475db936a0d217b9b6900aced8b3"  # Fester, geprüfter Implementierungsstand
MAX_HOURS = 6.0       # Obergrenze; Backups werden bereits nach jeder Episode gesichert
AUTO_RELEASE_GPU = True  # Erst nach bestätigtem Export auf Drive UND HF die VM freigeben
DRIVE_FOLDER = "/content/drive/MyDrive/LARA-Stack-Lab/stack-study-v2-32-hf"
HF_REPOSITORY = "Carlo0001/neura-lara5-cube-stacking-sim-v1"
HF_PREFIX = "backups/stack-study-v2-32-hf"
# Zum Fortsetzen denselben Commit, Drive-Ordner und HF-Pfad verwenden.
# Dieses HF-Notebook verwendet einen eigenen Ordner gegenüber dem älteren Lauf ohne HF-Backups.

## A100 prüfen und die festgelegte Umgebung einrichten

Die Python-Pakete der Physik sind für diesen Lauf festgelegt. Die Inferenz verwendet die bereits gesperrten GGUF-Dateien und llama.cpp-Revision. Beim ersten Einrichten kann auf älteren Colab-Systemen ein CUDA-Quellbuild erforderlich sein; dessen Log und Fortschritt werden ausgegeben.

In [ ]:
import os, sys, json, subprocess, signal, time, shutil, hashlib, urllib.request
from pathlib import Path
from datetime import datetime, timezone

def command(args, *, cwd=None, env=None, allowed=(0,)):
    print("$", " ".join(map(str,args)), flush=True)
    process = subprocess.Popen(list(map(str,args)), cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line,end="",flush=True)
        result = process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid,signal.SIGINT)
        try: process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGKILL)
            process.wait()
        raise
    finally:
        process.stdout.close()
    if result not in allowed:
        raise RuntimeError(f"Befehl endete mit {result}; Ausgabe oben prüfen.")
    return result

gpu_text = subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                                  "--format=csv,noheader,nounits"],text=True)
print(gpu_text)
gpu_rows = [row for row in gpu_text.splitlines() if row.strip()]
if len(gpu_rows) != 1 or "A100" not in gpu_rows[0]:
    raise RuntimeError("Für diesen Vergleich genau eine A100-Laufzeit auswählen.")
if not isinstance(MAX_HOURS,(int,float)) or not 0 < MAX_HOURS < float("inf"):
    raise ValueError("MAX_HOURS muss endlich und positiv sein.")

PROJECT=Path("/content/lara-stack-study-v2")
if not (PROJECT/".git").exists():
    PROJECT.mkdir(parents=True,exist_ok=True)
    command(["git","init"],cwd=PROJECT)
    command(["git","remote","add","origin","https://github.com/carlo2004/Simulation-Neura-LARA-5.git"],cwd=PROJECT)
    command(["git","fetch","--depth","1","origin",REPO_REF],cwd=PROJECT)
    command(["git","checkout","--detach","FETCH_HEAD"],cwd=PROJECT)
commit=subprocess.check_output(["git","rev-parse","HEAD"],cwd=PROJECT,text=True).strip()
if REPO_REF != "main" and commit != REPO_REF:
    raise RuntimeError("Der vorhandene Checkout ist ein anderer Stand. Neue Colab-Laufzeit verwenden.")
print("Software-Commit:",commit)
bootstrap_python=sys.executable
if sys.version_info[:2] not in ((3,12),(3,13)):
    command([sys.executable,"-m","pip","install","uv"])
    command([sys.executable,"-m","uv","python","install","3.12"])
    bootstrap_python=subprocess.check_output([sys.executable,"-m","uv","python","find","3.12"],text=True).strip()
PYTHON=PROJECT/".venv/bin/python"
ENV=dict(os.environ,OPENBLAS_NUM_THREADS="1",STACKLAB_CUDA_RUNTIME="auto",
         HF_HOME="/content/stacklab-hf-auth")
if not PYTHON.is_file():
    command([bootstrap_python,"-m","venv",PROJECT/".venv"],cwd=PROJECT)
command([PYTHON,"-m","pip","install","-r","requirements-study.txt"],cwd=PROJECT,env=ENV)
command([bootstrap_python,"bootstrap.py"],cwd=PROJECT,env=ENV)
def lab(*args,allowed=(0,)):
    return command([PYTHON,"-m","stacklab",*args],cwd=PROJECT,env=ENV,allowed=allowed)
lab("--version")

## Drive verbinden, HF-Zugriff testen und Checkpoints wiederherstellen

Der folgende Schritt meldet dich mit dem Secret/verdeckt eingegebenen Token an. Er schreibt eine kleine Testdatei auf HF, liest ihre Bytes am tatsächlichen Commit zurück und löscht sie wieder. Bei einem fehlgeschlagenen Test starten keine Modell-Downloads oder Simulationen.

Das voreingestellte Dataset ist das bereits im Projekt verwendete Repository. Ein bestehendes Dataset behält seine Sichtbarkeit; ein fehlendes wird privat angelegt. Konto/Repository und Sichtbarkeit werden beim Test angezeigt. Du kannst `HF_REPOSITORY` vorher anpassen.

Gesicherte Checkpoints werden von HF nach Drive zurückgeholt und per SHA-256 geprüft. Danach wird der 192-Versuche-Plan erstellt bzw. fortgesetzt. Immer nur eine Colab-Sitzung pro Studienordner verwenden. Die bereits geprüften 32 Startlagen lösen keine zusätzlichen Kontrollsimulationen aus.

In [ ]:
from google.colab import drive, userdata
from getpass import getpass

drive.mount("/content/drive")
try:
    hf_token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    hf_token = getpass("HF-Token mit Schreibzugriff (verdeckt): ")
if not hf_token or not hf_token.strip():
    raise ValueError("Ein HF-Token mit Schreibzugriff ist erforderlich.")
ENV["HF_TOKEN"] = hf_token.strip()
del hf_token
CHECKPOINT=Path(DRIVE_FOLDER)
WORK=Path("/content/stack-study-v2-hf-data")
HF_ARGS=("--repository",HF_REPOSITORY,"--prefix",HF_PREFIX)
lab("experiment","stack-study","hub-check",*HF_ARGS,"--checkpoint-dir",CHECKPOINT)
lab("experiment","stack-study","prepare","--output",WORK,"--checkpoint-dir",CHECKPOINT,
    "--hub-repo",HF_REPOSITORY,"--hub-prefix",HF_PREFIX)
# Keep the successful write-test receipt in the curated export as well.
(WORK/"sessions").mkdir(exist_ok=True)
for receipt in (CHECKPOINT/"sessions").glob("hf-write-test-*.json"):
    shutil.copyfile(receipt,WORK/"sessions"/receipt.name)
STUDY=json.loads((WORK/"study.json").read_text(encoding="utf-8"))
SETTINGS=STUDY["settings"]
MODELS=SETTINGS["models"]
print("Protokoll:",STUDY["protocol_sha256"])
print("Modelle:",MODELS)
print("Geplante Episoden:",len(STUDY["cases"])*len(MODELS))
print(json.loads((WORK/"collection-status.json").read_text(encoding="utf-8")))

## Nur die sechs ausgewählten Modelle installieren

Alle nutzen Q4_K_M und denselben Kontext von 16.384 Tokens. Herstellerprofile und Tokenizer bleiben als sichtbare Bedingungen gespeichert. Bis zu vier Programmblöcke pro Episode bekommen echte Ausführungsrückmeldungen. Modell-Fertigmeldungen ersetzen nicht die Kontakt- und Stabilitätsprüfung.

In [ ]:
recipe = "import json,sys; from stacklab.llm.catalog import lock; print(json.dumps([lock(m,device='cuda').get('cuda_runtime_kind') for m in sys.argv[1:]]))"
kinds=json.loads(subprocess.check_output([PYTHON,"-c",recipe,*MODELS],cwd=PROJECT,env=ENV,text=True))
if "source" in kinds:
    if not (PYTHON.parent/"cmake").is_file() and not shutil.which("cmake"):
        command([PYTHON,"-m","pip","install","cmake"],cwd=PROJECT,env=ENV)
    if not shutil.which("g++") or not shutil.which("make"):
        command(["apt-get","update","-qq"])
        command(["apt-get","install","-y","-qq","g++","make"])
    if not shutil.which("nvcc",path=ENV.get("PATH")):
        if not Path("/usr/local/cuda/bin/nvcc").is_file():
            raise RuntimeError("CUDA Toolkit fehlt; eine aktuelle Colab-GPU-Laufzeit verwenden.")
        ENV["PATH"]="/usr/local/cuda/bin:"+ENV.get("PATH","")
for model in MODELS:
    lab("setup","--model",model,"--device","cuda")

## Sammeln, laufend sichern, exportieren und GPU freigeben

Vor jeder Simulation wird ihre Startmarkierung nach Drive **und HF** gesichert. Nach jeder abgeschlossenen Episode wird ihr vollständiges Archiv hochgeladen und der HF-Commit bestätigt. Bereits gesicherte Archive werden übersprungen. Schlägt ein Upload nach drei Übertragungsversuchen fehl, stoppt die Erhebung vor weiteren ungesicherten Episoden; lokale und Drive-Dateien bleiben erhalten. Das sind Upload-Wiederholungen, keine zusätzlichen Modellversuche.

Nach `MAX_HOURS`, einem regulären Ende oder einem abgefangenen Fehler wird auch ein gesamter, checksummierter Export automatisch auf Drive und HF abgelegt. Die VM wird erst nach beiden erfolgreichen Exporten freigegeben.

Bei einem harten VM-Verlust sind die fertigen HF-Checkpoints unabhängig von der Colab-Laufzeit vorhanden. Die laufende Episode kann unvollständig bleiben; ihre gesicherte Startmarkierung verhindert ein versehentliches Wiederholen. Der nächste Start stellt die Aufzeichnungen wieder her und führt nur zuvor ungestartete Versuche aus. Eine Gesamtrangliste wird nicht erstellt.

In [ ]:
PORT=8765
URL=f"http://127.0.0.1:{PORT}"
server_log=Path("/content/stack-study-server.log")
server=None
export_verified=False
hub_export_verified=False
failure=None

def file_sha(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for part in iter(lambda:stream.read(1024*1024),b""):
            digest.update(part)
    return digest.hexdigest()

try:
    with server_log.open("w",encoding="utf-8") as log:
        server=subprocess.Popen([str(PYTHON),"-m","stacklab","serve","--device","cuda",
            "--model",MODELS[0],"--threads",str(SETTINGS["threads"]),"--port",str(PORT),
            "--context-tokens",str(SETTINGS["context_tokens"]),"--allow-model-switch","--install-missing"],
            cwd=PROJECT,env={key:value for key,value in ENV.items() if key not in
                ("HF_TOKEN","HUGGING_FACE_HUB_TOKEN")},stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        ready=False
        until=time.monotonic()+200
        opener=urllib.request.build_opener(urllib.request.ProxyHandler({}))
        while time.monotonic()<until:
            if server.poll() is not None:
                raise RuntimeError("Modellserver beendet sich: "+server_log.read_text(encoding="utf-8")[-8000:])
            try:
                with opener.open(URL+"/info",timeout=1) as response:
                    info=json.load(response)
                if info.get("hardware",{}).get("offloaded_layers",0)>0:
                    ready=True
                    break
            except OSError:
                pass
            time.sleep(.25)
        if not ready:
            raise RuntimeError("Server nicht bereit; "+str(server_log))
        lab("experiment","stack-study","run",WORK,"--server-url",URL,
            "--checkpoint-dir",CHECKPOINT,"--max-hours",str(MAX_HOURS),
            "--hub-repo",HF_REPOSITORY,"--hub-prefix",HF_PREFIX,allowed=(0,1))
except BaseException as error:
    failure=error
    print("Erhebung beendet/unterbrochen:",type(error).__name__,str(error),flush=True)
finally:
    if server is not None and server.poll() is None:
        os.killpg(server.pid,signal.SIGINT)
        try: server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(server.pid,signal.SIGKILL)
            server.wait()
    # Export even after an interrupted or technically failed collection.
    lab("experiment","stack-study","status",WORK)
    tag=datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    if server_log.exists():
        startup_record=dict(kind="model-server-startup",scored_model_data=False,
            error=str(failure) if failure else None,log=server_log.read_text(encoding="utf-8",errors="replace"))
        for folder in (WORK/"sessions",CHECKPOINT/"sessions"):
            folder.mkdir(parents=True,exist_ok=True)
            target=folder/("server-startup-"+tag+".json")
            temporary=target.with_suffix(".tmp")
            temporary.write_text(json.dumps(startup_record,ensure_ascii=False,indent=2),encoding="utf-8")
            temporary.replace(target)
    EXPORT=Path("/content")/("stack-study-v2-export-"+tag)
    lab("experiment","stack-study","export",WORK,"--output",EXPORT)
    DEST=CHECKPOINT/"exports"/EXPORT.name
    DEST.mkdir(parents=True,exist_ok=False)
    for source in EXPORT.iterdir():
        target=DEST/source.name
        temporary=target.with_suffix(target.suffix+".tmp")
        shutil.copyfile(source,temporary)
        if file_sha(source)!=file_sha(temporary):
            raise RuntimeError("Drive-Kopie stimmt nicht überein: "+source.name)
        temporary.replace(target)
    snapshot=json.loads((DEST/"snapshot.json").read_text(encoding="utf-8"))
    if file_sha(DEST/"evidence.zip") != snapshot["archive"]["sha256"]:
        raise RuntimeError("Prüfsumme des Drive-Exports stimmt nicht.")
    export_verified=True
    print("Export geprüft:",DEST,flush=True)
    try:
        lab("experiment","stack-study","hub-upload-export",EXPORT,*HF_ARGS)
        hub_export_verified=True
        print("Gesamtexport zusätzlich auf HF bestätigt.",flush=True)
    except Exception as error:
        if failure is None:
            failure=error
        print("Drive-Export ist gesichert; HF-Export fehlgeschlagen. VM bleibt zur Wiederherstellung offen.",flush=True)
    print("Inventar:",json.loads((DEST/"collection-status.json").read_text(encoding="utf-8")),flush=True)
    if AUTO_RELEASE_GPU and export_verified and hub_export_verified:
        print("Daten sind gesichert. Colab-GPU wird jetzt freigegeben.",flush=True)
        from google.colab import runtime
        try:
            runtime.unassign()
        except Exception as error:
            print("Automatische Freigabe fehlgeschlagen; Laufzeit bitte über Colab trennen:",str(error),flush=True)
if failure is not None:
    raise failure

## Nach Unterbrechung oder Abschluss

Für eine neue Colab-VM dieses Notebook erneut mit **denselben Einstellungen** starten. Login und Schreibtest laufen erneut, dann werden die geprüften HF-Checkpoints mit Drive zusammengeführt. Vorherige Modellversuche werden nicht ersetzt. Ein technischer Abbruch bleibt sichtbar; nur ungestartete Einträge werden ausgeführt. Es bleiben insgesamt 192 geplante Versuchseinheiten.

Die HF-Daten liegen unter `HF_PREFIX/batches/`; fertige Gesamtexporte unter `HF_PREFIX/exports/<snapshot_id>/`. Drive enthält ebenfalls die Einzelcheckpoints und Exportordner. `evidence.zip` enthält die vollständigen Anfragen, Antworten, Code, Feedback, Physik und Token-/Zeitmessungen. Token und Modellgewichte werden nicht mit hochgeladen.

Die 20 Stabilitätsproben und mehrere Programmblöcke erhöhen die Stichprobe nicht. 32 Szenen pro Modell sind ein erster gepaarter Vergleich; spätere Bestätigungs-/Fine-Tuning-Tests brauchen neue Startlagen.

[Versuchsprotokoll und Wiederherstellung](https://github.com/carlo2004/Simulation-Neura-LARA-5/blob/main/docs/stack-study-v2.md)